# Drug-Patient QGNN Training Notebook

This notebook provides a streamlined workflow for training the Quantum GNN model on drug-patient interaction data.

## Configuration

Modify the parameters below to customize your experiment.

In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import glob

In [ ]:
# TRAINING CONFIGURATION
# Data paths
DATA_DIR = os.getenv("DATA_DIR_local")  # Path to PDB data on SD card (READ-ONLY)
SAVE_DIR = "./saved_models"                      # Where to save models (LOCAL DISK)

# Data parameters
MAX_DRUGS = None            # Max drugs to load (None = all)
N_PATIENTS = 200             # Number of synthetic patients
INTERACTION_RATE = 0.05      # Fraction of drug-patient pairs to create

# Model parameters
NUM_QUBITS = 4               # Qubits per side (total = 2 * NUM_QUBITS)
NUM_QLAYERS = 2              # Number of variational layers
HIDDEN_DIM = 64              # Hidden dimension for encoders
USE_QUANTUM = True           # True: quantum mode, False: classical

# Training parameters
EPOCHS = 50                   # Number of training epochs
BATCH_SIZE = 10               # Batch size
LEARNING_RATE = 0.001        # Learning rate
VAL_SPLIT = 0.2              # Validation split (0.2 = 20%)
EARLY_STOPPING = None        # Early stopping patience (None = disabled)

# Other parameters
SEED = 42069                    # Random seed
DEVICE = None                # Device (None = auto-detect, 'cuda', 'mps', 'cpu')
VERBOSE = 2                  # Verbosity (0=silent, 1=progress, 2=detailed)

## Setup and Imports

In [ ]:




# Import drug-patient QGNN
from drug_patient_qgnn import (
    DrugPatientDataProcessor,
    QuantumDrugPatientGNN,
    DrugPatientTrainer,
    set_seed,
    print_model_summary,
    print_device_info,
    plot_training_history,
    calculate_metrics,
    print_metrics
)

# Set random seed
set_seed(SEED)

# Create save directory
os.makedirs(SAVE_DIR, exist_ok=True)

# Create experiment name
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
mode = "quantum" if USE_QUANTUM else "classical"
EXPERIMENT_NAME = f"{mode}_q{NUM_QUBITS}_l{NUM_QLAYERS}_{timestamp}"

print(f"Experiment: {EXPERIMENT_NAME}")
print("="*70)
print(f"Mode: {'QUANTUM' if USE_QUANTUM else 'CLASSICAL'}")
print(f"Qubits: {NUM_QUBITS} per side (total: {2*NUM_QUBITS})")
print(f"Layers: {NUM_QLAYERS}")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")


Experiment: quantum_q4_l2_20251116_100727
Mode: QUANTUM
Qubits: 4 per side (total: 8)
Layers: 2
Epochs: 50
Batch Size: 10
Learning Rate: 0.001


In [8]:
# Use real PDB data from SD card (read-only)
# No need to create sample descriptors - we have 15K+ real descriptor files!

print(f"Using real PDB data from: {DATA_DIR}")
print(f"The data will be loaded from existing descriptor CSV files during training.")
print(f"Results and models will be saved to: {SAVE_DIR}")

Using real PDB data from: /media/priyanshu/SD/othercode/data
The data will be loaded from existing descriptor CSV files during training.
Results and models will be saved to: ./saved_models


## Check Hardware

In [9]:
print_device_info()


Device Information
CUDA Available      : True
CUDA Devices        : 1
CUDA Device Name    : NVIDIA GeForce RTX 3080
MPS Available       : False



## Step 1: Load and Process Data

In [10]:
print("STEP 1: Loading Data")

# Initialize processor
processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)

# Load drug data from PDB descriptors
print(f"\nLoading drug data from: {DATA_DIR}")
n_drugs = processor.load_protein_ligand_data(max_samples=MAX_DRUGS)

# Fallback to synthetic if no data found
if n_drugs == 0:
    print("\nNo PDB data found. Creating synthetic drug data...")
    for i in range(50):
        drug_id = f"synthetic_drug_{i}"
        features = np.random.randn(11)
        processor.graph.add_drug(drug_id, features)
    print(f"Created {processor.graph.num_drugs()} synthetic drugs")

# Generate patients
print(f"\nGenerating {N_PATIENTS} synthetic patients...")
processor.create_synthetic_patient_data(n_patients=N_PATIENTS)

# Create interactions
print(f"\nGenerating interactions (rate={INTERACTION_RATE})...")
processor.create_synthetic_interactions(interaction_rate=INTERACTION_RATE)

# Get statistics
stats = processor.get_statistics()
print("Dataset Statistics:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"{key:25s}: {value:.4f}")
    else:
        print(f"{key:25s}: {value}")


STEP 1: Loading Data

Loading drug data from: /media/priyanshu/SD/othercode/data
Found 15239 descriptor files
Found 15239 descriptor files
Loaded 13201 drug nodes

Generating 200 synthetic patients...
Generating 200 synthetic patients...
Generated 200 patient nodes

Generating interactions (rate=0.05)...
Generating 126360 synthetic interactions...
Loaded 13201 drug nodes

Generating 200 synthetic patients...
Generating 200 synthetic patients...
Generated 200 patient nodes

Generating interactions (rate=0.05)...
Generating 126360 synthetic interactions...
Created 126360 interaction edges
Dataset Statistics:
num_drugs                : 12636
num_patients             : 200
num_interactions         : 126360
positive_rate            : 0.3027
negative_rate            : 0.6973
drug_feature_dim         : 19
patient_feature_dim      : 41
Created 126360 interaction edges
Dataset Statistics:
num_drugs                : 12636
num_patients             : 200
num_interactions         : 126360
positive_

## Step 2: Create Model

In [11]:
print("STEP 2: Creating Model")

# Get feature dimensions
graph = processor.graph
drug_dim = stats['drug_feature_dim']
patient_dim = stats['patient_feature_dim']

# Create model
model = QuantumDrugPatientGNN(
    drug_dim=drug_dim,
    patient_dim=patient_dim,
    num_qubits=NUM_QUBITS,
    num_qlayers=NUM_QLAYERS,
    hidden_dim=HIDDEN_DIM,
    use_quantum=USE_QUANTUM
)

# Print model summary
print_model_summary(model, drug_dim, patient_dim)

if USE_QUANTUM:
    print("\n  QUANTUM MODE ENABLED")
    print(f"  Total qubits: {2 * NUM_QUBITS}")
    print(f"  Drug qubits: 0-{NUM_QUBITS-1}")
    print(f"  Patient qubits: {NUM_QUBITS}-{2*NUM_QUBITS-1}")
else:
    print("\n🔷 CLASSICAL MODE (baseline)")

STEP 2: Creating Model

Model Summary
drug_dim            : 19
patient_dim         : 41
num_qubits          : 4
num_qlayers         : 2
use_quantum         : True
num_parameters      : 4841
Total parameters    : 4,841
Trainable params    : 4,841
Input (drug)        : (19,)
Input (patient)     : (41,)
Output              : (1,) [probability]


  QUANTUM MODE ENABLED
  Total qubits: 8
  Drug qubits: 0-3
  Patient qubits: 4-7


## Step 3: Train Model

In [ ]:
print("STEP 3: Training Model")

# Create trainer
trainer = DrugPatientTrainer(
    model,
    learning_rate=LEARNING_RATE,
    device=DEVICE
)

print(f"\nDevice: {trainer.device}")
print(f"Optimizer: Adam (lr={LEARNING_RATE})")
print(f"Loss: Binary Cross-Entropy")
print("\nStarting training...\n")

# Train model
history = trainer.fit(
    graph,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    verbose=VERBOSE,
    early_stopping_patience=EARLY_STOPPING
)

print("\n" + "="*70)
print("Training Complete!")
print("="*70)

STEP 3: Training Model

Device: cuda
Optimizer: Adam (lr=0.001)
Loss: Binary Cross-Entropy

Starting training...


Training on 101088 edges, validating on 25272 edges
Model: {'drug_dim': 19, 'patient_dim': 41, 'num_qubits': 4, 'num_qlayers': 2, 'use_quantum': True, 'num_parameters': 4841}
Device: cuda


Device: cuda
Optimizer: Adam (lr=0.001)
Loss: Binary Cross-Entropy

Starting training...


Training on 101088 edges, validating on 25272 edges
Model: {'drug_dim': 19, 'patient_dim': 41, 'num_qubits': 4, 'num_qlayers': 2, 'use_quantum': True, 'num_parameters': 4841}
Device: cuda

Epoch 1/50 - 10021.70s - train_loss: 0.6140 - train_acc: 0.6970 - val_loss: 0.6139 - val_acc: 0.6978 - val_auc: 0.5006
Epoch 1/50 - 10021.70s - train_loss: 0.6140 - train_acc: 0.6970 - val_loss: 0.6139 - val_acc: 0.6978 - val_auc: 0.5006


## Step 4: Visualize Training Progress

In [ ]:
# Plot training history
plot_training_history(history)

# Print final metrics
print("\nFinal Training Metrics:")
print(f"Train Loss:     {history['train_loss'][-1]:.4f}")
print(f"Train Accuracy: {history['train_acc'][-1]:.4f}")
print(f"Val Loss:       {history['val_loss'][-1]:.4f}")
print(f"Val Accuracy:   {history['val_acc'][-1]:.4f}")
print(f"Val AUC-ROC:    {history['val_auc'][-1]:.4f}")


## Step 5: Evaluate Model

In [ ]:
print("\n" + "="*70)
print("STEP 5: Model Evaluation")
print("="*70)

# Get predictions on all data
drug_features = torch.tensor(graph.get_drug_features_matrix(), dtype=torch.float32)
patient_features = torch.tensor(graph.get_patient_features_matrix(), dtype=torch.float32)

edge_index, _ = graph.get_edge_index()
drug_indices = edge_index[0]
patient_indices = edge_index[1]

model.eval()
with torch.no_grad():
    edge_drug_features = drug_features[drug_indices].to(trainer.device)
    edge_patient_features = patient_features[patient_indices].to(trainer.device)
    predictions = model(edge_drug_features, edge_patient_features).cpu().numpy().flatten()

# Get true labels
true_labels = graph.get_edge_labels()

# Calculate comprehensive metrics
pred_labels = (predictions >= 0.5).astype(int)
metrics = calculate_metrics(true_labels, pred_labels, predictions)

# Print metrics
print_metrics(metrics, title="Model Performance Metrics")

## Step 6: Visualize Predictions

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Confusion Matrix
cm = confusion_matrix(true_labels, pred_labels)
im = axes[0].imshow(cm, cmap='Blues', aspect='auto')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Predicted Fail', 'Predicted Success'])
axes[0].set_yticklabels(['True Fail', 'True Success'])
axes[0].set_title('Confusion Matrix')

for i in range(2):
    for j in range(2):
        text = axes[0].text(j, i, cm[i, j], ha="center", va="center", 
                           color="red", fontsize=20, fontweight='bold')

plt.colorbar(im, ax=axes[0])

# ROC Curve
fpr, tpr, _ = roc_curve(true_labels, predictions)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', lw=2, 
            label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

# Prediction Distribution
axes[2].hist(predictions[true_labels == 0], bins=20, alpha=0.5, 
            label='Actual Fail', color='red')
axes[2].hist(predictions[true_labels == 1], bins=20, alpha=0.5, 
            label='Actual Success', color='green')
axes[2].axvline(x=0.5, color='black', linestyle='--', label='Threshold')
axes[2].set_xlabel('Predicted Probability')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Prediction Distribution')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_evaluation.png"), dpi=150)
plt.show()

## Step 7: Save Model and Results

In [ ]:
print("\n" + "="*70)
print("STEP 7: Saving Results")
print("="*70)

# Save model state
model_path = os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_model.pt")
torch.save(model.state_dict(), model_path)
print(f"✓ Model saved: {model_path}")

# Save checkpoint
checkpoint_path = os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_checkpoint.pt")
trainer.save_checkpoint(checkpoint_path)
print(f"✓ Checkpoint saved: {checkpoint_path}")

# Save graph
graph_path = os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_graph.pkl")
processor.save_graph(graph_path)
print(f"✓ Graph saved: {graph_path}")

# Save training history
from drug_patient_qgnn import save_training_history
history_path = os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_history.json")
save_training_history(history, history_path)
print(f"✓ History saved: {history_path}")

# Save configuration
import json
config = {
    'experiment_name': EXPERIMENT_NAME,
    'timestamp': timestamp,
    'data': {
        'data_dir': DATA_DIR,
        'max_drugs': MAX_DRUGS,
        'n_patients': N_PATIENTS,
        'interaction_rate': INTERACTION_RATE,
    },
    'model': {
        'num_qubits': NUM_QUBITS,
        'num_qlayers': NUM_QLAYERS,
        'hidden_dim': HIDDEN_DIM,
        'use_quantum': USE_QUANTUM,
    },
    'training': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'val_split': VAL_SPLIT,
    },
    'results': {
        'final_train_loss': float(history['train_loss'][-1]),
        'final_train_acc': float(history['train_acc'][-1]),
        'final_val_loss': float(history['val_loss'][-1]),
        'final_val_acc': float(history['val_acc'][-1]),
        'final_val_auc': float(history['val_auc'][-1]),
    },
    'metrics': {k: float(v) if isinstance(v, (int, float, np.number)) else v 
                for k, v in metrics.items()}
}

config_path = os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_config.json")
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✓ Config saved: {config_path}")

print("\n" + "="*70)
print(f"All results saved to: {SAVE_DIR}")
print("="*70)

## Step 8: Test Prediction on New Samples

In [ ]:
print("\n" + "="*70)
print("STEP 8: Testing Predictions")
print("="*70)

# Select random samples
n_samples = 5
sample_indices = np.random.choice(len(drug_indices), n_samples, replace=False)

print(f"\nPredicting {n_samples} random drug-patient pairs:\n")

model.eval()
with torch.no_grad():
    for idx in sample_indices:
        drug_idx = drug_indices[idx]
        patient_idx = patient_indices[idx]
        
        drug_id = list(graph.drug_nodes.keys())[drug_idx]
        patient_id = list(graph.patient_nodes.keys())[patient_idx]
        
        # Get features
        drug_feat = drug_features[drug_idx:drug_idx+1].to(trainer.device)
        patient_feat = patient_features[patient_idx:patient_idx+1].to(trainer.device)
        
        # Predict
        pred_prob = model(drug_feat, patient_feat).item()
        true_label = int(true_labels[idx])
        
        print(f"Pair {idx+1}:")
        print(f"  Drug: {drug_id[:30]}...")
        print(f"  Patient: {patient_id}")
        print(f"  Predicted Success Prob: {pred_prob:.2%}")
        print(f"  Prediction: {'SUCCESS' if pred_prob >= 0.5 else 'FAILURE'}")
        print(f"  True Label: {'SUCCESS' if true_label == 1 else 'FAILURE'}")
        print(f"  Correct: {'✓' if (pred_prob >= 0.5) == (true_label == 1) else '✗'}")
        print()

print("="*70)

## Summary

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT SUMMARY")
print("="*70)
print(f"Experiment Name: {EXPERIMENT_NAME}")
print(f"Mode: {'Quantum' if USE_QUANTUM else 'Classical'}")
print(f"\nData:")
print(f"  Drugs: {stats['num_drugs']}")
print(f"  Patients: {stats['num_patients']}")
print(f"  Interactions: {stats['num_interactions']}")
print(f"\nModel:")
print(f"  Qubits: {NUM_QUBITS} × 2 = {2*NUM_QUBITS}")
print(f"  Layers: {NUM_QLAYERS}")
print(f"  Parameters: {model.get_num_parameters():,}")
print(f"\nTraining:")
print(f"  Epochs: {len(history['train_loss'])}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"\nFinal Results:")
print(f"  Train Loss: {history['train_loss'][-1]:.4f}")
print(f"  Train Acc:  {history['train_acc'][-1]:.4f}")
print(f"  Val Loss:   {history['val_loss'][-1]:.4f}")
print(f"  Val Acc:    {history['val_acc'][-1]:.4f}")
print(f"  Val AUC:    {history['val_auc'][-1]:.4f}")
print(f"\nTest Metrics:")
print(f"  Accuracy:   {metrics['accuracy']:.4f}")
print(f"  Precision:  {metrics['precision']:.4f}")
print(f"  Recall:     {metrics['recall']:.4f}")
print(f"  F1 Score:   {metrics['f1']:.4f}")
print(f"  AUC-ROC:    {metrics['auc_roc']:.4f}")
print(f"\nSaved to: {SAVE_DIR}/")
print("="*70)

print("\n✅ Training Complete!")

## Optional: Load and Test Saved Model

In [ ]:
# Uncomment to test loading the saved model

# # Create new model with same architecture
# loaded_model = QuantumDrugPatientGNN(
#     drug_dim=drug_dim,
#     patient_dim=patient_dim,
#     num_qubits=NUM_QUBITS,
#     num_qlayers=NUM_QLAYERS,
#     hidden_dim=HIDDEN_DIM,
#     use_quantum=USE_QUANTUM
# )

# # Load weights
# loaded_model.load_state_dict(torch.load(model_path))
# loaded_model.eval()

# print("✓ Model loaded successfully!")

# # Test prediction
# with torch.no_grad():
#     test_drug = drug_features[0:1]
#     test_patient = patient_features[0:1]
#     test_pred = loaded_model(test_drug, test_patient)
#     print(f"Test prediction: {test_pred.item():.4f}")